# AI Engineering Interview Prep
## Section: Python

**Mangesh Jha | Senior Software Developer → AI Engineer**

---
> 💡 **How to use:** Use `Ctrl+F` to search any question. 🏷️ markers link to Mangesh's real project experience.


In [ ]:
# ── Optional: Install dependencies for runnable code cells ────────────────
# Uncomment and run this cell first if you want to execute the code examples

# !pip install sentence-transformers faiss-cpu numpy pydantic langchain
# !pip install langchain-community langchain-google-genai
# !pip install groq openai anthropic
# !pip install ragas datasets evaluate

print("📚 AI Engineering Interview Prep Notebook")
print("✅ Ready to study! All 511+ questions are answered below.")
print()
print("Sections:")
sections = [
    "1. LLM Fundamentals (63 Qs)",
    "2. Prompt Engineering (30 Qs)",
    "3. RAG (37 Qs)",
    "4. AI Agents (37 Qs)",
    "5. Fine-Tuning (25 Qs)",
    "6. Vector Databases (22 Qs)",
    "7. AI System Design (42 Qs)",
    "8. LLMOps (41 Qs)",
    "9. Evaluation (29 Qs)",
    "10. AI Safety & Ethics (38 Qs)",
    "11. Multimodal AI (26 Qs)",
    "12. AI Infrastructure (27 Qs)",
    "13. Coding Practicals (22 Qs)",
    "14. Behavioral Questions (22 Qs)",
    "15. LangChain (10 Qs)",
    "16. LangGraph (10 Qs)",
    "17. Python (10 Qs)",
    "18. FastAPI (10 Qs)",
    "19. Resume-Based Questions (10 Qs)",
    "20. Quick Reference Cheat Sheet",
]
for s in sections:
    print(f"  📌 Section {s}")


---
## 🐍 Section 17 — Python

### Q1. How do generators and `yield` work? How are they useful in AI applications?
**A:** Standard functions run to completion and return a single result. A generator function uses `yield` to return a value, suspend its execution, and save its state. When called again, it resumes right where it left off. In AI engineering, memory is a major constraint. If you're processing a corpus of 10 million documents for RAG, loading them all into a Python list will crash your container (OOM). With a generator, you stream documents from disk or DB one by one, embed them in batches, and write to the vector DB. You only keep one batch in memory at any time.

### Q2. How do decorators work in Python? How do you write a decorator that accepts arguments?
**A:** Decorators are wrappers around functions (or classes) that allow you to execute code before and after the wrapped function runs, without modifying its source. To write a decorator that accepts arguments (like a retry decorator with a `max_retries` count), you need three levels of functions:
1. The outer function accepts the decorator arguments.
2. The middle function (the actual decorator) accepts the target function.
3. The inner function (the wrapper) accepts the target function's arguments, runs the logic (e.g., a try-except loop up to `max_retries`), and returns the result.
Using `@functools.wraps` is essential to preserve the original function's name and docstring.

### Q3. Explain the context manager protocol and how it applies to resource management?
**A:** Context managers ensure resources are cleaned up properly (like files, database sessions, or GPU memory), even if exceptions occur. You use them with the `with` statement. Under the hood, Python calls `__enter__` to set up the resource and `__exit__` to clean it up when the block finishes. You can also write them easily using the `@contextmanager` decorator from `contextlib` with a `try/finally` block. In AI, they are vital for managing PyTorch GPU sessions (`with torch.no_grad():`) to disable gradient calculations and free up VRAM, or database connections in RAG pipelines.

### Q4. What is the GIL (Global Interpreter Lock)? How does it affect multi-threading vs. multi-processing vs. asyncio?
**A:** The GIL is a mutex in CPython that ensures only one thread executes Python bytecode at a time. This prevents race conditions in memory management but makes multi-threading useless for CPU-bound tasks (like calculating embeddings or matrix multiplications in Python), as they won't run on multiple CPU cores.
- **For CPU-bound tasks:** Use `multiprocessing` to spawn separate OS processes with their own Python interpreters and memory spaces, bypassing the GIL.
- **For I/O-bound tasks (like LLM API calls, fetching web pages):** Use `asyncio` or `threading`. The thread/task yields execution while waiting for the network, letting other threads run.

### Q5. How does `asyncio` achieve concurrency under the hood?
**A:** `asyncio` achieves concurrency using cooperative multitasking driven by an event loop. Unlike OS-level multi-threading (where the OS preemptively switches threads), `asyncio` runs in a single thread. When a function hits an `await` statement (like a database query or an LLM call), it yields control back to the event loop, which immediately schedules another task that is ready to run. This is incredibly lightweight. Spawning 10,000 OS threads will crash your server, but spawning 10,000 asyncio tasks uses minimal memory because there is no thread context-switching overhead.

### Q6. What are metaclasses, and how do they differ from class decorators?
**A:** A class decorator takes a fully constructed class and modifies it. A metaclass is a "class of a class"—it defines *how* a class is constructed in the first place (it overrides `__new__` or `__init__` of the `type` class). You use class decorators for simple post-construction modifications. You use metaclasses when you need deep, structural control over class creation, inheritance, or field registration. For example, Pydantic uses metaclasses to parse field annotations and compile validation schemas before the class is ever instantiated.

### Q7. How does Python manage memory, and how do you prevent leaks in RAG systems?
**A:** Python uses reference counting (deleting objects when their reference count hits 0) combined with a generational Garbage Collector to catch cyclic references (e.g., Object A references B, and B references A). In RAG systems, memory leaks often happen when storing massive document chunks in long-lived variables (like global lists, caches, or state variables in class instances). To prevent this:
1. Avoid global state.
2. Use generator streams to process data.
3. Explicitly clear large data structures (`del my_list` or `my_list.clear()`).
4. Use weak references (`weakref`) if you need to keep references without increasing the reference count.

### Q8. Explain dunder (magic) methods and when you would implement them.
**A:** Dunder (double underscore) methods allow custom classes to hook into Python's built-in syntax.
- `__call__`: Makes an instance callable like a function (e.g., LangChain's Runnables implement `__call__` or `invoke`).
- `__getitem__` / `__setitem__`: Allows indexing like a list or dictionary (`my_obj[key]`).
- `__getattr__` / `__setattr__`: Customizes attribute access.
- `__enter__` / `__exit__`: Implements the context manager protocol.
You implement them to build clean, intuitive APIs for other developers in your team.

### Q9. Why is type hinting critical in AI codebases? What are Annotated, Union, and Generic?
**A:** LLMs return unpredictable structures, and RAG pipelines pass around complex objects. Without type hints, debugging runtime type errors is a nightmare. Using `mypy` helps catch errors during CI/CD before code runs.
- `Annotated`: Attaches metadata to types (e.g., `Annotated[str, "metadata info"]` used in LangGraph to define reducers like `Annotated[list, add_messages]`).
- `Union` (or `|` in Python 3.10+): Specifies that a value can be one of multiple types.
- `Generic`: Allows you to write reusable container classes (like a custom `Response[T]` class) that maintain type checking regardless of the wrapped object's type.

### Q10. Compare Python dataclasses vs. Pydantic models. When do you use each?
**A:** 
- **Dataclasses (standard library):** Great for internal data containers. They generate boilerplate (`__init__`, `__repr__`) automatically. They do *not* perform runtime validation by default; they only document types. They are lightweight and fast.
- **Pydantic Models:** Perform strict runtime data validation and serialization. If a field type doesn't match the schema, it raises a validation error. Essential for API payloads (FastAPI), parsing structured outputs from LLMs, and validating complex configurations.
**Rule:** Use dataclasses for internal, trusted data structures. Use Pydantic for system boundaries (APIs, LLM outputs, user input).